<a href="https://colab.research.google.com/github/RiznoFadhil/ui-greenmetric-sustainability-analysis/blob/main/WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from time import sleep
import re

In [9]:
# =========================
# COLUMN MAPPING
# =========================
COLUMN_MAPPING = {
    "Rank": ["Rank", "No", "Ranking", "World Rank", "Rank 2020", "Rank 2021", "Rank 2022"],
    "University": ["University", "University Name"],
    "Country": ["Country", "Country Name"],
    "Total Score": ["Total", "Total Score", "Overall Score"],

    "Setting & Infrastructure": ["SI Score", "Setting & Infrastructure", "SI"],
    "Energy & Climate Change": ["EC Score", "Energy & Climate Change"],
    "Waste": ["WS Score", "Waste"],
    "Water": ["WR Score", "Water"],
    "Transportation": ["TR Score", "Transportation"],
    "Education & Research": ["ED Score", "Education & Research"]
}

In [10]:
# =========================
# STANDARDIZE COLUMN NAMES
# =========================
def standardize_columns(df, column_mapping):
    new_columns = {}
    for col in df.columns:
        col_clean = str(col).strip()
        for standard_name, variants in column_mapping.items():
            if col_clean in variants:
                new_columns[col] = standard_name
                break
    return df.rename(columns=new_columns)

In [12]:
import pandas as pd
from time import sleep

years = [2020, 2021, 2022]
base_url = "https://greenmetric.ui.ac.id/rankings/overall-rankings-{}"

all_years = []

# Define rank column variants for identification
RANK_COL_VARIANTS = COLUMN_MAPPING["Rank"]

for year in years:
    print(f"Scraping {year}")

    try:
        # Get all tables from the URL
        list_of_dfs = pd.read_html(base_url.format(year))

        found_df = None
        for temp_df in list_of_dfs:
            # Check if any of the rank column variants exist in the current dataframe's columns
            # Ensure columns are treated as strings for comparison to avoid errors
            temp_df_cols_str = [str(col).strip() for col in temp_df.columns]

            if any(variant in temp_df_cols_str for variant in RANK_COL_VARIANTS):
                found_df = temp_df
                break

        if found_df is None:
            raise ValueError(f"No table containing a 'Rank' column variant found for {year}")

        df = found_df

        # Ensure all column names are strings before standardization
        # This prevents the "'int' object has no attribute 'strip'" error
        df.columns = [str(col) for col in df.columns]

        # Standardize renamed columns
        df = standardize_columns(df, COLUMN_MAPPING)

        # Add Year column
        df["Year"] = year

        # Ensure consistent column order
        final_columns = [
            "Year",
            "Rank",
            "University",
            "Country",
            "Total Score",
            "Setting & Infrastructure",
            "Energy & Climate Change",
            "Waste",
            "Water",
            "Transportation",
            "Education & Research"
        ]

        # Filter and reorder columns, handling potential missing columns gracefully
        # Create a list of columns that actually exist in df after standardization
        existing_final_columns = [col for col in final_columns if col in df.columns]
        df = df[existing_final_columns]

        # Add any missing final_columns with pd.NA values if they were not in the scraped data
        for col in final_columns:
            if col not in df.columns:
                df[col] = pd.NA # Use pd.NA for missing values

        # Reorder to ensure final_columns order
        df = df[final_columns]


        all_years.append(df)

        df.to_excel(f"UI_GreenMetric_Overall_{year}.xlsx", index=False)

        print(f"✓ {year} done")
        sleep(2)

    except Exception as e:
        print(f"✗ {year} failed: {e}")

# Combine all years
combined_df = pd.concat(all_years, ignore_index=True)
combined_df.to_excel(
    "UI_GreenMetric_Overall_Rankings_2020_2022.xlsx", # Changed filename
    index=False
)

print("🎉 All years standardized and saved")

Scraping 2020
✓ 2020 done
Scraping 2021
✓ 2021 done
Scraping 2022
✓ 2022 done
🎉 All years standardized and saved
